In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px

day = 0
file_name = f"./round-3-island-data-bottle/prices_round_3_day_{day}.csv"
df0 = pd.read_csv(file_name, sep=';')

day = 1
file_name = f"./round-3-island-data-bottle/prices_round_3_day_{day}.csv"
df1 = pd.read_csv(file_name, sep=';')

day = 2
file_name = f"./round-3-island-data-bottle/prices_round_3_day_{day}.csv"
df2 = pd.read_csv(file_name, sep=';')


In [ ]:
df = pd.concat([df0, df1])

In [ ]:
fig = go.Figure()
df_gift_basket = df[df['product'] == 'GIFT_BASKET'].copy()
df_gift_basket['swmid'] = (df_gift_basket['bid_price_1'] * df_gift_basket['ask_volume_1'] + df_gift_basket['ask_price_1'] * df_gift_basket['bid_volume_1']) / (df_gift_basket['ask_volume_1'] + df_gift_basket['bid_volume_1'])
fig.add_trace(go.Scatter(x=df.index, y=df_gift_basket['swmid'], mode='lines', name='Gift Basket Price', line=dict(color='blue')))
fig.update_layout(title_text="Gift Basket Price")
fig.update_xaxes(title_text="Index")
fig.update_yaxes(title_text="Price")
fig.show()


In [ ]:
import pandas as pd
day = 1
file_name = f"./round-3-island-data-bottle/prices_round_3_day_{day}.csv"
df2 = pd.read_csv(file_name, sep=';')
df = df2

In [ ]:
from lib import parse_log_file
import numpy as np
import json
import pandas as pd
df, df_trades, _ = parse_log_file(f"./clean_data_logs/trade_history_day_{day}.log")

In [ ]:
df_chocolate = df[df['product'] == 'CHOCOLATE'].copy()
df_strawberries = df[df['product'] == 'STRAWBERRIES'].copy()
df_roses = df[df['product'] == 'ROSES'].copy()
df_gift_basket = df[df['product'] == 'GIFT_BASKET'].copy()
df_chocolate['swmid'] = (df_chocolate['bid_price_1'] * df_chocolate['ask_volume_1'] + df_chocolate['ask_price_1'] * df_chocolate['bid_volume_1']) / (df_chocolate['ask_volume_1'] + df_chocolate['bid_volume_1'])
df_strawberries['swmid'] = (df_strawberries['bid_price_1'] * df_strawberries['ask_volume_1'] + df_strawberries['ask_price_1'] * df_strawberries['bid_volume_1']) / (df_strawberries['ask_volume_1'] + df_strawberries['bid_volume_1'])
df_roses['swmid'] = (df_roses['bid_price_1'] * df_roses['ask_volume_1'] + df_roses['ask_price_1'] * df_roses['bid_volume_1']) / (df_roses['ask_volume_1'] + df_roses['bid_volume_1'])
df_gift_basket['swmid'] = (df_gift_basket['bid_price_1'] * df_gift_basket['ask_volume_1'] + df_gift_basket['ask_price_1'] * df_gift_basket['bid_volume_1']) / (df_gift_basket['ask_volume_1'] + df_gift_basket['bid_volume_1'])
df_synthetic = pd.DataFrame({
    'timestamp': df_chocolate['timestamp'].to_numpy(),
    'bid_price_1': df_chocolate['bid_price_1'].to_numpy() * 4 + df_strawberries['bid_price_1'].to_numpy() * 6 + df_roses['bid_price_1'].to_numpy(),
    'bid_volume_1': np.min(np.array([df_chocolate['bid_volume_1'].to_numpy(), df_strawberries['bid_volume_1'].to_numpy(), df_roses['bid_volume_1'].to_numpy()]), axis=0),
    'ask_price_1': df_chocolate['ask_price_1'].to_numpy() * 4 + df_strawberries['ask_price_1'].to_numpy() * 6 + df_roses['ask_price_1'].to_numpy(),
    'ask_volume_1': np.min(np.array([df_chocolate['ask_volume_1'].to_numpy(), df_strawberries['ask_volume_1'].to_numpy(), df_roses['ask_volume_1'].to_numpy()]), axis=0),
    'mid_price':  df_chocolate['mid_price'].to_numpy() * 4 + df_strawberries['mid_price'].to_numpy() * 6 + df_roses['mid_price'].to_numpy(),
    'swmid': df_chocolate['swmid'].to_numpy() * 4 + df_strawberries['swmid'].to_numpy() * 6 + df_roses['swmid'].to_numpy()
    })


In [ ]:
from plotly.subplots import make_subplots

fig = make_subplots(specs=[[{"secondary_y": True}]])
# fig.add_trace(go.Scatter(x=df_synthetic['timestamp'], y=df_synthetic['swmid'] + 370, mode='lines', name='Synthetic SWMID', line=dict(color='green')))
fig.add_trace(go.Scatter(x=df_gift_basket['timestamp'], y=df_gift_basket['swmid'], mode='lines', name='Gift Basket SWMID', line=dict(color='blue')), secondary_y=False)
fig.add_trace(go.Scatter(x=df_synthetic['timestamp'], y=df_synthetic['swmid'], mode='lines', name='Synthetic SWMID', line=dict(color='red')), secondary_y=True)
fig.update_layout(title_text="Synthetic and Gift Basket SWMID")
fig.update_xaxes(title_text="Timestamp")
fig.update_yaxes(title_text="SWMID", secondary_y=False)
fig.update_yaxes(title_text="Synthetic SWMID", secondary_y=True)
fig.show()




In [ ]:

fig = go.Figure()
spread = pd.DataFrame({'timestamp': df_synthetic['timestamp'], 'spread': df_gift_basket['swmid'].to_numpy() - df_synthetic['swmid'].to_numpy()})

# fig.add_trace(go.Scatter(x=df_synthetic['timestamp'], y=spread, mode='lines', name='Difference', line=dict(color='red')))

spread['std30'] = spread['spread'].rolling(window=50).std()
spread['sma'] = spread['spread'].rolling(window=300).mean()



fig.add_trace(go.Scatter(x=spread.index, y=spread['spread'], mode='lines', name='Spread', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=spread.index, y=spread['sma'], mode='lines', name='sma', line=dict(color='red')))
spread['z_score'] = (spread['spread'] - spread['sma']) / spread['std30']


fig.add_trace(go.Scatter(x=spread.index, y=spread['z_score'], mode='lines', name='Z Score', line=dict(color='purple'), yaxis='y2'))
fig.update_layout(yaxis2=dict(title='Z Score', overlaying='y', side='right'))
fig.update_layout(title_text="Spread and Moving Averages")
fig.update_xaxes(title_text="Index")
fig.update_yaxes(title_text="Value")


In [ ]:
from tqdm import tqdm
spread['std30'] = spread['spread'].rolling(window=30).std()
z_score = (spread['spread'].to_numpy() - 376) / spread['std30'].to_numpy()
spread_market = pd.DataFrame({'timestamp': df_synthetic['timestamp'].to_numpy(), 'swmid': df_gift_basket['swmid'].to_numpy() - df_synthetic['swmid'].to_numpy()})

def cross_spread(cash, quantity):
    return cash - abs(quantity) * 10

def backtest(thresh, target_position, std_window, sma_window, verbose=False):
    cash = 0
    position = 0
    pnl_hist = []
    position_hist = []
    cash_hist = []
    spread[f'std{std_window}'] = spread['spread'].rolling(window=std_window).std()
    spread[f'sma{sma_window}'] = spread['spread'].rolling(window=sma_window).mean()
    z_score = (spread['spread'].to_numpy() - spread[f'sma{sma_window}']) / spread[f'std{std_window}'].to_numpy()
    spread_market['spread_z'] = z_score
    for index, row in spread_market.iterrows():
        if index == 0:
            continue
        swmid = row['swmid']
        
        if row['spread_z'] > thresh and position != -target_position:
        
            
            quantity = -target_position - position
            cash -= (-target_position - position) * swmid
            cash = cross_spread(cash, quantity)
            position = -target_position
            
            if verbose:
                print(f"SELL {quantity} AT PRICE {swmid} AT TIME {row['timestamp']}")
        
        if row['spread_z'] < -thresh and position != target_position:
            quantity = target_position - position
            cash -= (target_position - position) * swmid
            cash = cross_spread(cash, quantity)
            position = target_position
            
            if verbose:
                print(f"BUY {quantity} FOR PRICE {swmid} AT TIME {row['timestamp']}")
    
        position_hist.append(position)
        cash_hist.append(cash)
        pnl_hist.append(cash + position * swmid)
        
    if verbose:
        print(f"PNL: {pnl_hist[-1]}")
        
    return pnl_hist

In [ ]:
position_opt = [60]
thresh_opt = [1,2,3,5,6,7,7.5,8,9,10,15,20,25]
std_window_opt = [10,20,25,30,35,40,50]
sma_window_opt = [10,20,25,30,35,40,50,75, 100, 125, 150, 200, 300, 500]
opt = []
for thresh in tqdm(thresh_opt): 
    for std_window in std_window_opt: 
        for sma_window in sma_window_opt:
            for position in position_opt:
                pnl = backtest(thresh, position, std_window, sma_window)
                opt.append({"thresh": thresh, "position": position, "std_window": std_window, "sma_window": sma_window, "pnl": pnl})
#                 print("="*80)
#                 print(f"Thresh: {thresh}, Position: {position}, Std Window: {std_window}, PnL: {pnl[-1]}")
#                 print("="*80)

In [ ]:
spread_market = pd.DataFrame({'timestamp': df_synthetic['timestamp'].to_numpy(), 'spread_z': z_score, 'swmid': df_gift_basket['swmid'].to_numpy() - df_synthetic['swmid'].to_numpy()})

In [ ]:
opt.sort(key=lambda x: x['pnl'][-1], reverse=True)
top_3_pnl = opt[:3]
pnl_graph = top_3_pnl[0]['pnl']

fig = go.Figure()
fig.add_trace(go.Scatter(x=spread_market.index, y=pnl_graph, mode='lines', name='PnL'))
fig.show()


In [ ]:
top_3_pnl = opt[:3]
top_pnl_params = top_3_pnl[0]
for params in top_3_pnl:
    print(f"thresh: {params['thresh']}, std_window: {params['std_window']}, pnl: {params['pnl'][-1]}")

In [ ]:
pnl_params = [{**{k: v for k, v in d.items() if k != "pnl"}, 'pnl': d['pnl'][-1]} for d in opt]

In [ ]:
pnl_params

In [ ]:
res = backtest(top_pnl_params['thresh'], top_pnl_params['position'], top_pnl_params['std_window'], verbose = True)

# backtest with clearing

In [ ]:
from tqdm import tqdm
spread['std30'] = spread['spread'].rolling(window=30).std()
z_score = (spread['spread'].to_numpy() - 376) / spread['std30'].to_numpy()
spread_market = pd.DataFrame({'timestamp': df_synthetic['timestamp'].to_numpy(), 'swmid': df_gift_basket['swmid'].to_numpy() - df_synthetic['swmid'].to_numpy()})

def cross_spread(cash, quantity):
    return cash - abs(quantity) * 10

def backtest(take_thresh,clear_thresh, target_position, std_window, verbose=False):
    cash = 0
    position = 0
    pnl_hist = []
    position_hist = []
    cash_hist = []
    spread[f'std{std_window}'] = spread['spread'].rolling(window=std_window).std()
    z_score = (spread['spread'].to_numpy() - 376) / spread[f'std{std_window}'].to_numpy()
    spread_market['spread_z'] = z_score
    for index, row in spread_market.iterrows():
        if index == 0:
            continue
        swmid = row['swmid']
        
        if row['spread_z'] > take_thresh and position != -target_position:
            quantity = -target_position - position
            cash -= (-target_position - position) * swmid
            cash = cross_spread(cash, quantity)
            position = -target_position
            
            if verbose:
                print(f"SELL {quantity} AT PRICE {swmid} AT TIME {row['timestamp']}")
        
        if row['spread_z'] < -take_thresh and position != target_position:
            quantity = target_position - position
            cash -= (target_position - position) * swmid
            cash = cross_spread(cash, quantity)
            position = target_position
            
            if verbose:
                print(f"BUY {quantity} FOR PRICE {swmid} AT TIME {row['timestamp']}")
            
        if (row['spread_z'] < clear_thresh and row['spread_z'] > -clear_thresh) and position != 0:
            quantity = -position
            cash -= (quantity * swmid)
            cash = cross_spread(cash, quantity)
            position = 0
            
            if verbose:
                print(f"CLEAR {quantity} FOR PRICE {swmid} AT TIME {row['timestamp']}")
    
        position_hist.append(position)
        cash_hist.append(cash)
        pnl_hist.append(cash + position * swmid)
    if verbose:
        print(f"PNL: {pnl_hist[-1]}")
    return pnl_hist

In [ ]:
position_opt = [60]
thresh_opt = [1,2,3,5,7.5,10,15,20,25,30,35, 40, 45]
clear_thresh_opt = [0,0.5,1,2,3,5,6,7,7.5,8,9,10,15,20,25]
std_window_opt = [10,20,25,30,35,40,50]
opt_clear = []
for thresh in tqdm(thresh_opt): 
    for clear_thresh in [t for t in clear_thresh_opt if t <= thresh]:
        for std_window in std_window_opt: 
            for position in position_opt:
                pnl = backtest(thresh, clear_thresh, position, std_window)
                opt_clear.append({"thresh": thresh, "clear_thresh": clear_thresh, "position": position, "std_window": std_window, "pnl": pnl})

In [ ]:
opt_clear.sort(key=lambda x: x['pnl'][-1], reverse=True)
top_3_pnl = opt_clear[:3]
pnl_graph = top_3_pnl[0]['pnl']

fig = go.Figure()
fig.add_trace(go.Scatter(x=spread_market.index, y=pnl_graph, mode='lines', name='PnL'))
fig.show()


In [ ]:
top_pnl_params = top_3_pnl[0]

In [ ]:
pnl_params = [{k: v for k, v in d.items() if k != "pnl"} for d in opt_clear]

In [ ]:
res = backtest(top_pnl_params['thresh'], top_pnl_params['clear_thresh'], top_pnl_params['position'], top_pnl_params['std_window'], verbose = True)

In [ ]:
pnl_params

In [ ]:
params = {'thresh': 15, 'clear_thresh': 0.5, 'position': 60, 'std_window': 10}
res = backtest(params['thresh'], params['clear_thresh'], params['position'], params['std_window'], verbose = True)

In [ ]:

fig = go.Figure()
spread = pd.DataFrame({'timestamp': df_synthetic['timestamp'], 'spread': df_gift_basket['swmid'].to_numpy() - df_synthetic['swmid'].to_numpy()})

# fig.add_trace(go.Scatter(x=df_synthetic['timestamp'], y=spread, mode='lines', name='Difference', line=dict(color='red')))

spread['std30'] = spread['spread'].rolling(window=params['std_window']).std()
# spread['std30'] = spread['spread'].rolling(window=100).apply(lambda x: np.sqrt(np.mean((x-376)**2)))



fig.add_trace(go.Scatter(x=spread.index, y=spread['spread'], mode='lines', name='Spread', line=dict(color='blue')))
# fig.add_trace(go.Scatter(x=spread.index, y=spread['sma5'], mode='lines', name='SMA5', line=dict(color='green')))
# fig.add_trace(go.Scatter(x=spread.index, y=spread['sma60'], mode='lines', name='SMA60', line=dict(color='red')))
spread['z_score'] = (spread['spread'] - 376) / spread['std30']
fig.add_shape(
    type="line",
    x0=spread.index[0],
    y0=params['thresh'],
    x1=spread.index[-1],
    y1=params['thresh'],
    line=dict(
        color="LightSeaGreen",
        width=2,
        dash="dashdot",
    ),
    yref="y2"
)

fig.add_shape(
    type="line",
    x0=spread.index[0],
    y0=-params['thresh'],
    x1=spread.index[-1],
    y1=-params['thresh'],
    line=dict(
        color="LightSeaGreen",
        width=2,
        dash="dashdot",
    ),
    yref="y2"
)


fig.add_shape(
    type="line",
    x0=spread.index[0],
    y0=params['clear_thresh'],
    x1=spread.index[-1],
    y1=params['clear_thresh'],
    line=dict(
        color="LightBlue",
        width=1,
        dash="dashdot",
    ),
    yref="y2"
)

fig.add_shape(
    type="line",
    x0=spread.index[0],
    y0=-params['clear_thresh'],
    x1=spread.index[-1],
    y1=-params['clear_thresh'],
    line=dict(
        color="LightBlue",
        width=1,
        dash="dashdot",
    ),
    yref="y2"
)


fig.add_trace(go.Scatter(x=spread.index, y=spread['z_score'], mode='lines', name='Z Score', line=dict(color='purple'), yaxis='y2'))
fig.update_layout(yaxis2=dict(title='Z Score', overlaying='y', side='right'))
fig.update_layout(title_text="Spread and Moving Averages")
fig.update_xaxes(title_text="Index")
fig.update_yaxes(title_text="Value")
